<a href="https://colab.research.google.com/github/takuonakashima/ai-security-workshop/blob/main/generate_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

# 1. AIの構造を定義
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

# 2. 学習用データの準備
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# 3. 学習の準備（GPUの設定、誤差関数の定義）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 学習ループ（1エポックだけ回す：約1〜2分）
print("学習を開始します...")
model.train()
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    if batch_idx % 100 == 0:
        print(f"進捗: [{batch_idx * len(data)}/60000]  Loss: {loss.item():.6f}")

# 5. 完成した脳みそ（重み）をファイルとして保存
torch.save(model.state_dict(), 'mnist_cnn.pth')
print("✅ 学習完了！ 'mnist_cnn.pth' を保存しました。")

100%|██████████| 9.91M/9.91M [00:00<00:00, 36.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 964kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 8.59MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.92MB/s]


学習を開始します...
進捗: [0/60000]  Loss: 2.298632
進捗: [6400/60000]  Loss: 0.089417
進捗: [12800/60000]  Loss: 0.287080
進捗: [19200/60000]  Loss: 0.017260
進捗: [25600/60000]  Loss: 0.152662
進捗: [32000/60000]  Loss: 0.074943
進捗: [38400/60000]  Loss: 0.047973
進捗: [44800/60000]  Loss: 0.039372
進捗: [51200/60000]  Loss: 0.060884
進捗: [57600/60000]  Loss: 0.060416
✅ 学習完了！ 'mnist_cnn.pth' を保存しました。
